# When implementing linear regression (and just using NumPy in general), we perform a lot of matrix inversions

> But how does this work in practice?

In [ ]:
import numpy as np

## Now let's talk analysis

The only abstraction here is matrix inversion, so let's implement it and understand the math.

Assume $A$ is invertible. If we can decompose $A = LU$, where $L$ is lower triangular and $U$ is upper triangular, then we can use it to solve the linear system $AX = I$ ($X$ would be $A^{-1}$).

For numerical stability, use LU with partial pivoting to swap rows to ensure the largest magnitude pivot is being used each step. This avoids a lot of numerical stability from eliminating sall values.

**Input:** Square matrix $ A \in \mathbb{R}^{n \times n} $  
**Output:** Permutation matrix $ P $, lower triangular $ L $, upper triangular $ U $ such that $ PA = LU $

---

1. Initialize:
   - $ P \leftarrow I_n $
   - $ L \leftarrow 0_{n \times n} $
   - $ U \leftarrow A $

2. For $ k = 1 $ to $ n $:

   a. **Pivot selection:**
   - Find row  
     $$
     p = \arg\max_{i = k, \dots, n} |U[i, k]|
     $$

   b. **Row swap (if needed):**
   - If $ p \neq k $:
     - Swap rows $ k $ and $ p $ in $ U $
     - Swap rows $ k $ and $ p $ in $ P $
     - Swap rows $ k $ and $ p $ in $ L[1:k-1, :] $

   c. **Elimination:**
   - For $ i = k+1 $ to $ n $:
     - $ L[i, k] \leftarrow \frac{U[i, k]}{U[k, k]} $
     - For $ j = k $ to $ n $:
       $$
       U[i, j] \leftarrow U[i, j] - L[i, k] \cdot U[k, j]
       $$

3. Set diagonal of $ L $:
   - For $ i = 1 $ to $ n $:
     - $ L[i, i] \leftarrow 1 $

---

**Return:** $ P, L, U $

In [12]:
def LU_partial_pivot(A):
  n = A.shape[0]
  P = np.eye(n, dtype='float64')
  L = np.eye(n, dtype='float64')
  U = np.copy(A)

  for k in range(n):
    p = k + np.argmax(np.abs(U[k:, k]))
    if p != k: # row swap
      U[[k, p]] = U[[p, k]]
      P[[k, p]] = P[[p, k]]
      L[[k, p], :k] = L[[p, k], :k]

    for i in range(k+1, n):
      L[i,k] = U[i,k] / U[k,k]
      U[i, k:] -= L[i, k] * U[k, k:]

  return P, L, U

In [13]:
A = np.array([[1,2,3], [4,5,6], [7,8,9]])
A = A.astype(float)
P, L, U = LU_partial_pivot(A)

In [14]:
A.shape

(3, 3)

In [15]:
L @ U

array([[7., 8., 9.],
       [1., 2., 3.],
       [4., 5., 6.]])

In [16]:
P @ A

array([[7., 8., 9.],
       [1., 2., 3.],
       [4., 5., 6.]])

## Now that we can solve the LU decomposition, how is this useful?

First, for a permutation matrix $P^T = P^{-1}$

So $PA = LU \implies A = P^T LU$

Then $A^{-1} = U^{-1}L^{-1}P$

Solve for $U^{-1}$ with back substitution


In [17]:
def back_substitution(A, y):
  """Assume A upper triangular"""
  n = A.shape[0]
  b = np.zeros(n)
  for i in range(n-1, -1, -1):

    target = y[i] - np.dot(A[i, i+1:], b[i+1:])
    b[i] = target / A[i,i]

  return b

def forward_substitution(A, y):
  """Assume A lower triangular"""
  n = A.shape[0]
  b = np.zeros(n)
  for i in range(n):

    target = y[i] - np.dot(A[i, :i], b[:i])
    b[i] = target / A[i,i]

  return b

In [18]:
A = np.array([[1,2,3], [0,2,3], [0,0,3]])
z = np.array([1,2,3])
b = back_substitution(A, z)

In [19]:
A @ b

array([1., 2., 3.])

In [20]:
A = np.array([[1,0,0], [1,2,0], [1,2,3]])
z = np.array([1,2,3])
b = forward_substitution(A, z)

In [21]:
A @ b

array([1., 2., 3.])

In [22]:
def invert_upper_tri(A):
  n = A.shape[0]
  A_inv = np.zeros((n, n))
  for i in range(n):
    y = np.zeros(n)
    y[i] = 1
    col = back_substitution(A, y)
    A_inv[:, i] = col

  return A_inv

In [23]:
A = np.array([[1,2,3], [0,2,3], [0,0,3]])
A_inv = invert_upper_tri(A)

In [24]:
A @ A_inv

array([[ 1.00000000e+00,  0.00000000e+00, -5.55111512e-17],
       [ 0.00000000e+00,  1.00000000e+00, -5.55111512e-17],
       [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00]])

In [25]:
def invert_lower_tri(A):
  n = A.shape[0]
  A_inv = np.zeros((n, n))
  for i in range(n):
    y = np.zeros(n)
    y[i] = 1
    col = forward_substitution(A, y)
    A_inv[:, i] = col

  return A_inv

In [26]:
A = np.array([[1,0,0], [1,2,0], [1,2,3]])
A_inv = invert_lower_tri(A)
A @ A_inv

array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [27]:
def invert_matrix(A):
  P, L, U = LU_partial_pivot(A)

  L_inv = invert_lower_tri(L)
  U_inv = invert_upper_tri(U)
  return U_inv @ L_inv @ P

In [28]:
A = np.array([[1,2,3], [4,5,6], [7,8,9]], dtype='float64')
A_inv = invert_matrix(A)

In [29]:
A_inv @ A

array([[3., 0., 1.],
       [1., 8., 7.],
       [0., 0., 0.]])